In [ ]:
import pandas as pd

manif = pd.read_csv("adni_data/MPR-R,_GradWarp,_B1,_N3,_Scaled_Manifest_14Aug2025.csv")
db = pd.read_csv("adni_data/All_Subjects_My_Table_11Aug2025.csv")

# Filter to matching subject_ids and make a copy
patients_diagnosis = db[db['subject_id'].isin(manif['subject_id'])].copy()

cols = ["BCSTROKE", "EXNEURO", "EXNIMAG", "EXILLNESS", "EXLAB"]

# Convert columns to numeric
patients_diagnosis[cols] = patients_diagnosis[cols].apply(pd.to_numeric, errors='coerce')

# Drop rows where any column == 1
filtered_df = patients_diagnosis[~(patients_diagnosis[cols] == 1).any(axis=1)]
len(filtered_df)

In [5]:
for col in patients_diagnosis.columns:
    print(f"\n--- {col} ---")
    print(patients_diagnosis[col].value_counts(dropna=False))



--- subject_id ---
subject_id
003_S_0908    28
126_S_0605    28
027_S_0074    27
041_S_0679    27
003_S_1122    27
              ..
141_S_0340     1
014_S_0357     1
027_S_1335     1
067_S_0020     1
067_S_0024     1
Name: count, Length: 675, dtype: int64

--- visit ---
visit
bl        658
sc        658
m06       627
m12       596
m18       567
m24       530
m30       404
m36       381
m48       221
m42       216
m72       195
m60       192
m66       168
m78       167
m54       166
m84       159
m96       120
m90        97
m108       93
AUT        63
nv         61
m120       58
m144       37
m132       32
m156       23
m168       20
m180       20
m150       17
f          17
m138       15
m162       15
4_init     15
m126       12
m186       12
m102        8
m174        8
m192        7
m198        7
m222        6
m204        5
m114        5
m216        5
m234        4
m228        3
m210        3
4_m12       2
uns1        1
4_sc        1
Name: count, dtype: int64

--- PTGENDER ---
PTGEND

In [6]:
for col in filtered_df.columns:
    print(f"\n--- {col} ---")
    print(filtered_df[col].value_counts(dropna=False))



--- subject_id ---
subject_id
126_S_0605    28
003_S_0908    28
027_S_0074    27
003_S_1122    27
041_S_0679    27
              ..
094_S_0964     1
033_S_0888     1
027_S_0948     1
128_S_0805     1
027_S_1335     1
Name: count, Length: 674, dtype: int64

--- visit ---
visit
bl        658
sc        646
m06       627
m12       594
m18       567
m24       528
m30       404
m36       377
m48       219
m42       216
m72       194
m60       191
m66       168
m78       167
m54       166
m84       157
m96       120
m90        97
m108       92
AUT        63
nv         61
m120       57
m144       37
m132       32
m156       22
m168       20
m180       20
m150       17
f          16
m138       15
m162       15
4_init     15
m126       12
m186       12
m102        8
m174        8
m192        7
m198        7
m222        6
m204        5
m114        5
m216        5
m234        4
m228        3
m210        3
4_m12       2
uns1        1
4_sc        1
Name: count, dtype: int64

--- PTGENDER ---
PTGEND

In [7]:
patients_diagnosis['PTGENDER'].value_counts()

PTGENDER
1.0    3638
2.0    3059
Name: count, dtype: int64

In [8]:
patients_diagnosis['subject_id'].nunique()

675

In [9]:
# Identify male rows (assuming PTGENDER: 1 = male, 2 = female)
male_rows = filtered_df[filtered_df['PTGENDER'] == 1]

# Randomly select 600 male rows to drop
drop_males = male_rows.sample(n=600, random_state=42).index

# Drop them from the dataframe
balanced_df = filtered_df.drop(drop_males)

# Verify counts
print(balanced_df['PTGENDER'].value_counts())


PTGENDER
2.0    3041
1.0    3026
Name: count, dtype: int64


In [10]:
balanced_df = balanced_df[balanced_df["CENROLL"] != 0]


In [11]:
balanced_df['subject_id'].nunique()

658

In [12]:
balanced_df.columns

Index(['subject_id', 'visit', 'PTGENDER', 'ARM', 'BCSTROKE', 'IHSYMPTOM',
       'MHPSYCH', 'MH2NEURL', 'EXNEURO', 'EXNIMAG', 'EXILLNESS', 'EXLAB',
       'CENROLL'],
      dtype='object')

In [13]:
balanced_df.drop(columns=cols, inplace=True)

In [14]:
manif.head()

,image_id,subject_id,image_date,image_description
0,143389,127_S_1427,2009-03-12,MPR-R; GradWarp; B1 Correction; N3
1,91130,127_S_1427,2007-08-20,MPR-R; GradWarp; B1 Correction; N3
2,99271,127_S_1427,2008-02-19,MPR-R; GradWarp; B1 Correction; N3
3,121698,127_S_1427,2008-09-02,MPR-R; GradWarp; B1 Correction; N3
4,163865,127_S_1427,2009-08-26,MPR-R; GradWarp; B1 Correction; N3


In [15]:
missing_in_balanced = manif[~manif['subject_id'].isin(balanced_df['subject_id'])]
print(missing_in_balanced.count())


image_id             23
subject_id           23
image_date           23
image_description    23
dtype: int64


In [16]:
extra_in_balanced = balanced_df[~balanced_df['subject_id'].isin(manif['subject_id'])]
print(extra_in_balanced)


Empty DataFrame
Columns: [subject_id, visit, PTGENDER, ARM, IHSYMPTOM, MHPSYCH, MH2NEURL, CENROLL]
Index: []


In [17]:
balanced_df.to_csv("MRI_MANIFEST_CLEANED.csv")

In [18]:
import pandas as pd

df = pd.read_csv("adni_data/MRI_MANIFEST_CLEANED.csv")

In [19]:
df.head()

,Unnamed: 0,subject_id,visit,PTGENDER,ARM,IHSYMPTOM,MHPSYCH,MH2NEURL,CENROLL
0,87,003_S_1122,4_init,2.0,5.0,NaN,NaN,NaN,NaN
1,88,003_S_1122,4_m12,2.0,5.0,NaN,NaN,NaN,NaN
2,89,003_S_1122,bl,2.0,5.0,NaN,NaN,NaN,NaN
3,90,003_S_1122,m06,2.0,5.0,NaN,NaN,NaN,NaN
4,91,003_S_1122,m108,2.0,5.0,NaN,NaN,NaN,NaN


In [20]:
print(",".join(df['subject_id'].unique().astype(str)))


003_S_1122,128_S_0200,114_S_0416,073_S_0089,003_S_0908,007_S_1222,041_S_0679,011_S_0021,126_S_0605,141_S_0767,057_S_0934,002_S_1280,127_S_1427,002_S_0413,002_S_1155,011_S_0002,011_S_0003,022_S_0004,011_S_0005,022_S_0007,011_S_0008,022_S_0014,011_S_0016,067_S_0019,011_S_0022,067_S_0029,023_S_0030,023_S_0031,035_S_0033,099_S_0040,007_S_0041,023_S_0042,022_S_0044,011_S_0023,018_S_0043,067_S_0038,067_S_0045,011_S_0010,035_S_0048,099_S_0051,011_S_0053,067_S_0056,018_S_0055,099_S_0054,067_S_0059,023_S_0058,099_S_0060,023_S_0061,022_S_0066,007_S_0068,023_S_0078,067_S_0076,023_S_0083,067_S_0077,023_S_0084,027_S_0074,023_S_0093,099_S_0090,022_S_0096,007_S_0101,020_S_0097,023_S_0081,130_S_0102,099_S_0111,127_S_0112,023_S_0126,041_S_0125,022_S_0130,027_S_0116,023_S_0139,131_S_0123,007_S_0070,098_S_0149,007_S_0128,018_S_0142,067_S_0110,098_S_0160,027_S_0118,128_S_0138,035_S_0156,128_S_0135,128_S_0167,027_S_0120,022_S_0129,098_S_0171,011_S_0168,032_S_0187,128_S_0188,114_S_0166,114_S_0173,067_S_0098

In [7]:
import os
import pandas as pd

id_set = set()
dir = "adni_data/MRIs_IMAGES_FILTERED_NOMASKS"
imgs = os.listdir(dir)
len(imgs)

756

In [8]:

for fname in imgs:
    id = fname.replace("ADNI_","").split("_MR_MPR-R_")[0]
    id_set.add(id)

In [9]:
id_set

{'002_S_0413',
 '002_S_0685',
 '002_S_0816',
 '002_S_0938',
 '002_S_0954',
 '002_S_1070',
 '002_S_1155',
 '002_S_1268',
 '002_S_1280',
 '003_S_0908',
 '003_S_0931',
 '003_S_1074',
 '003_S_1122',
 '005_S_0222',
 '005_S_0223',
 '005_S_0324',
 '005_S_0448',
 '005_S_0546',
 '005_S_0553',
 '005_S_0572',
 '005_S_0602',
 '005_S_0610',
 '005_S_1224',
 '006_S_0498',
 '006_S_0547',
 '006_S_0675',
 '006_S_0681',
 '006_S_0731',
 '006_S_1130',
 '007_S_0041',
 '007_S_0068',
 '007_S_0070',
 '007_S_0101',
 '007_S_0128',
 '007_S_0249',
 '007_S_0293',
 '007_S_0344',
 '007_S_1206',
 '007_S_1222',
 '007_S_1304',
 '011_S_0005',
 '011_S_0008',
 '011_S_0016',
 '011_S_0021',
 '011_S_0022',
 '011_S_0023',
 '011_S_0053',
 '011_S_0168',
 '011_S_0241',
 '011_S_0362',
 '011_S_0856',
 '011_S_0861',
 '011_S_1080',
 '011_S_1282',
 '013_S_0240',
 '013_S_0325',
 '013_S_0502',
 '013_S_0575',
 '013_S_0996',
 '013_S_1035',
 '013_S_1186',
 '013_S_1205',
 '013_S_1275',
 '014_S_0169',
 '014_S_0328',
 '014_S_0356',
 '014_S_05

In [10]:
len(id_set)

456